In [ ]:
import pandas as pd
import re

# Load the initial datasets
file_paths = [
    "/mnt/data/Economic Data Release Dates Investing.com  - Jan 2024.csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet19 (1).csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet20 (1).csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet16.csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet17.csv",
    "/mnt/data/Economic Data Release Dates Investing.com  - Sheet18.csv"
]

# Read all files into dataframes and concatenate them into one dataframe
combined_df = pd.concat([pd.read_csv(file_path) for file_path in file_paths], ignore_index=True)

# Forward fill the missing dates
combined_df['Date'] = combined_df['Date'].ffill()

# Combine the 'Date' and 'Time' columns into a single 'DateTime' column and convert to pandas datetime format
combined_df['DateTime'] = pd.to_datetime(combined_df['Date'] + ' ' + combined_df['Time'], errors='coerce')

# Drop rows where DateTime could not be parsed
combined_df = combined_df.dropna(subset=['DateTime'])

# Sort the dataframe by DateTime in ascending order
combined_df_sorted = combined_df.sort_values(by='DateTime').reset_index(drop=True)

# Select relevant columns for the final dataset
final_combined_df = combined_df_sorted[['DateTime', 'Event']]

In [ ]:
# Define the list of specific events we want to extract with exact matches ignoring parenthesis content
events_of_interest_exact = [
    "Atlanta Fed GDPNow",
    "Average Hourly Earnings",
    "CPI",
    "Core CPI",
    "Core CPI Index",
    "Core Durable Goods Orders",
    "Core PCE Price Index",
    "Core Retail Sales",
    "Durable Goods Orders",
    "Fed Interest Rate Decision",
    "ISM Manufacturing PMI",
    "ISM Non-Manufacturing PMI",
    "Nonfarm Payrolls",
    "PCE Price index",
    "Retail Sales",
    "Unemployment Rate",
    "FOMC Press Conference"
]

# Function to match events ignoring content in parenthesis
def match_event_with_parenthesis(event_name):
    event_name_cleaned = re.sub(r'\s*\(.*?\)\s*', '', event_name).strip()
    for target_event in events_of_interest_exact:
        if target_event.lower() == event_name_cleaned.lower():
            return True
    return False

# Filter the dataframe using the refined matching function
filtered_final_df_corrected = final_combined_df[final_combined_df['Event'].apply(match_event_with_parenthesis)]

# Select only the relevant columns for the final dataframe
result_final_df_corrected = filtered_final_df_corrected[['DateTime', 'Event']].dropna().reset_index(drop=True)

# Load the signal dataframe
signal_dataset_path = "/mnt/data/Signature_AI_Results_Final.csv"
signal_df = pd.read_csv(signal_dataset_path)

# Convert 'Datetime' column in the signal dataframe to pandas datetime format
signal_df['Datetime'] = pd.to_datetime(signal_df['Datetime'], errors='coerce')

# Merge the two dataframes on the date part of the datetime column
signal_df['Date'] = signal_df['Datetime'].dt.date
result_final_df_corrected['Date'] = result_final_df_corrected['DateTime'].dt.date

# Merge the two dataframes on the date column
merged_df_by_date_with_datetime = pd.merge(signal_df, result_final_df_corrected, left_on='Date', right_on='Date', how='left')

# Group by 'Datetime' in the merged dataframe and concatenate the events and event datetimes
merged_df_by_date_with_datetime['events'] = merged_df_by_date_with_datetime.groupby('Datetime')['Event'].transform(lambda x: ', '.join(x.dropna().unique()))
merged_df_by_date_with_datetime['event_datetimes'] = merged_df_by_date_with_datetime.groupby('Datetime')['DateTime'].transform(lambda x: ', '.join(pd.to_datetime(x.dropna()).dt.strftime('%Y-%m-%d %H:%M:%S').unique()))

# Drop the extra 'DateTime', 'Event', and 'Date' columns
merged_df_by_date_with_datetime = merged_df_by_date_with_datetime.drop(columns=['DateTime', 'Event', 'Date']).drop_duplicates()

# Save the updated signal dataframe to a CSV file
updated_signal_final_csv_path = "/mnt/data/Updated_Signal_Dataset_Final.csv"
merged_df_by_date_with_datetime.to_csv(updated_signal_final_csv_path, index=False)


In [ ]:
# Load the signal dataframe
signal_dataset_path = "E:\Signal Backtesting\Input\sigai_signals_h.csv"
signal_df = pd.read_csv(signal_dataset_path)

# Convert 'Datetime' column in the signal dataframe to pandas datetime format
signal_df['Datetime'] = pd.to_datetime(signal_df['Datetime'], errors='coerce')

# Merge the two dataframes on the date part of the datetime column
signal_df['Date'] = signal_df['Datetime'].dt.date
result_final_df_corrected['Date'] = result_final_df_corrected['DateTime'].dt.date

# Merge the two dataframes on the date column
merged_df_by_date_with_datetime = pd.merge(signal_df, result_final_df_corrected, left_on='Date', right_on='Date', how='left')

# Group by 'Datetime' in the merged dataframe and concatenate the events and event datetimes
merged_df_by_date_with_datetime['events'] = merged_df_by_date_with_datetime.groupby('Datetime')['Event'].transform(lambda x: ', '.join(x.dropna().unique()))
merged_df_by_date_with_datetime['event_datetimes'] = merged_df_by_date_with_datetime.groupby('Datetime')['DateTime'].transform(lambda x: ', '.join(pd.to_datetime(x.dropna()).dt.strftime('%Y-%m-%d %H:%M:%S').unique()))

# Drop the extra 'DateTime', 'Event', and 'Date' columns
merged_df_by_date_with_datetime = merged_df_by_date_with_datetime.drop(columns=['DateTime', 'Event', 'Date']).drop_duplicates()

# Save the updated signal dataframe to a CSV file
updated_signal_final_csv_path = "/mnt/data/Updated_Signal_Dataset_Final.csv"
merged_df_by_date_with_datetime.to_csv(updated_signal_final_csv_path, index=False)

In [ ]:
import pandas as pd

# Load the datasets
merged_dataset = pd.read_csv('/mnt/data/Merged_Dataset_Economic.csv')
important_events = pd.read_csv('/mnt/data/important_events_Dataset.csv')

# Define the main names to search for
main_names = [
    "Existing Home Sales", 
    "Existing Home Sales (MoM)",
    "New Home Sales (Dec)",
    "New Home Sales (MoM) (Dec)"
]

# Function to check if any of the main names are in the event name, ignoring additional parenthesis content
def is_important_event(event_name):
    for name in main_names:
        base_name = name.split(' (')[0]
        if base_name in event_name:
            return True
    return False

# Filter the events that match the criteria
important_events_filtered = merged_dataset[merged_dataset['Event'].apply(is_important_event)]

# Add the filtered events to the important events dataset
updated_important_events = pd.concat([important_events, important_events_filtered], ignore_index=True)

# Save the updated dataset to a new CSV file
updated_important_events.to_csv('/mnt/data/updated_important_events_Dataset_v2.csv', index=False)


In [2]:
import pandas as pd
# Load the datasets
updated_important_events = pd.read_csv('E:\Signal Backtesting\Input\\updated_important_events_Dataset_v2.csv')
signal_dataset = pd.read_csv('E:\Signal Backtesting\Input\sigai_signals_h.csv')

# Ensure the DateTime columns are in datetime format
updated_important_events['DateTime'] = pd.to_datetime(updated_important_events['DateTime'])
signal_dataset['Datetime'] = pd.to_datetime(signal_dataset['Datetime'])

# Function to add events to the signal dataset without repeating events
def add_unique_events_to_signal(signal_df, events_df):
    # Iterate over each event and add it to the signal dataset where the dates match
    for idx, event in events_df.iterrows():
        event_date = event['DateTime'].date()
        event_name = event['Event']
        event_datetime = str(event['DateTime'])
        
        for index, row in signal_df[signal_df['Datetime'].dt.date == event_date].iterrows():
            existing_events = row['events'] if pd.notna(row['events']) else ''
            existing_event_datetimes = row['event_datetimes'] if pd.notna(row['event_datetimes']) else ''
            
            if event_name not in existing_events:
                new_events = (existing_events + ', ' + event_name).strip(' ,')
                new_event_datetimes = (existing_event_datetimes + ', ' + event_datetime).strip(' ,')
                
                signal_df.at[index, 'events'] = ', '.join(sorted(set(new_events.split(', '))))
                signal_df.at[index, 'event_datetimes'] = ', '.join(sorted(set(new_event_datetimes.split(', '))))
                
    return signal_df

# Update the signal dataset with the important events without repeating them
updated_signal_dataset = add_unique_events_to_signal(signal_dataset, updated_important_events)

# Save the updated dataset to a new CSV file
updated_signal_dataset.to_csv('E:\Signal Backtesting\Input\\Updated_sigai_signals_h_with_Important_Events.csv', index=False)


KeyError: 'events'

In [ ]:
import pandas as pd

# Load the new dataset
file_path = 'E:\Signal Backtesting\Input\Economic Data Release Dates Investing.com  -July.csv'
df_july = pd.read_csv(file_path)

# Forward fill the missing dates
df_july['Date'] = df_july['Date'].fillna(method='ffill')

# Combine the Date and Time columns
df_july['Datetime'] = pd.to_datetime(df_july['Date'] + ' ' + df_july['Time'], format='%A, %B %d, %Y %H:%M')

# Save the modified dataframe to a new CSV file
output_file_path = 'E:\Signal Backtesting\Input\Economic Data Release Dates Investing.com  -July.csv'
df_july.to_csv(output_file_path, index=False)

In [2]:
import pandas as pd
import datetime
import re


# Load the datasets
economic_data_path = 'E:\Signal Backtesting\Input\Economic Data Release Dates Investing.com  -July.csv'
signal_data_path = 'E:\Signal Backtesting\Input\\updated_signal_dataset_with_events.csv'

economic_data = pd.read_csv(economic_data_path)
signal_data = pd.read_csv(signal_data_path)

# Convert Datetime columns to datetime objects
economic_data['Datetime'] = pd.to_datetime(economic_data['Datetime'])
signal_data['Datetime'] = pd.to_datetime(signal_data['Datetime'])

# Extract the date part
economic_data['Date'] = economic_data['Datetime'].dt.date
signal_data['Date'] = signal_data['Datetime'].dt.date

# Ensure no leading or trailing whitespace in event names
economic_data['Event'] = economic_data['Event'].str.strip()

# List of core event words to include
main_names = [
    "Atlanta Fed GDPNow", "Average Hourly Earnings (MoM)", "CPI (MoM)", "CPI (YoY)",
    "Core CPI (MoM)", "Core CPI (YoY)", "Core CPI Index", "Core Durable Goods Orders (MoM)",
    "Core PCE Price Index (MoM)", "Core PCE Price Index (YoY)", "Core Retail Sales (MoM)",
    "Durable Goods Orders (MoM)", "Fed Interest Rate Decision", "ISM Manufacturing PMI",
    "ISM Non-Manufacturing PMI", "Nonfarm Payrolls", "PCE Price index (YoY)",
    "Retail Sales (MoM)", "Unemployment Rate", "FOMC Press Conference", "Core PPI (MoM) (Jun)",
    "Core PPI (YoY) (Jun)","PPI (MoM) (Jun)", "PPI (YoY) (Jun)"
]


# Filter economic data to keep only the events that match the main event names
def is_important_event(event_name):
    event_base_name = event_name.split(' (')[0].strip()
    for name in main_names:
        main_base_name = name.split(' (')[0].strip()
        if event_base_name == main_base_name:
            return True
    return False


economic_data_filtered = economic_data[economic_data['Event'].apply(is_important_event)]

# Filter signal data to include only dates in July 2024
#signal_data_filtered = signal_data[
    #(signal_data['Date'] >= datetime.date(2024, 7, 1)) & (signal_data['Date'] <= datetime.date(2024, 7, 31))]

# Merge datasets on the date
merged_data = pd.merge(signal_data, economic_data_filtered[['Date', 'Event', 'Datetime']], on='Date',how='left')

# Group by the signal data rows and concatenate the events and datetimes, including repeated datetimes
merged_data['events'] = merged_data.groupby(['Datetime_x', 'value', 'rolling_std', 'rolling_mean', 'Signal'])[
    'Event'].transform(lambda x: ', '.join(x.dropna()))
merged_data['event_datetimes'] = merged_data.groupby(['Datetime_x', 'value', 'rolling_std', 'rolling_mean', 'Signal'])[
    'Datetime_y'].transform(lambda x: ', '.join(x.dropna().astype(str)))

# Drop duplicates
final_data = merged_data.drop_duplicates(
    subset=['Datetime_x', 'value', 'rolling_std', 'rolling_mean', 'Signal', 'events', 'event_datetimes'])

# Rename columns to match the original signal data
final_data.rename(columns={'Datetime_x': 'Datetime'}, inplace=True)

# Drop unnecessary columns
final_data.drop(columns=['Date', 'Event', 'Datetime_y'], inplace=True)

# Save the final dataset
final_data_path = 'E:\Signal Backtesting\Input\\updated_signal_dataset_with_repeated_event_datetimes.csv'
final_data.to_csv(final_data_path, index=False)

C:\Users\Zeinab\AppData\Local\Temp\ipykernel_5040\1412942546.py:66: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data.rename(columns={'Datetime_x': 'Datetime'}, inplace=True)
C:\Users\Zeinab\AppData\Local\Temp\ipykernel_5040\1412942546.py:69: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_data.drop(columns=['Date', 'Event', 'Datetime_y'], inplace=True)
